# Satu pembanding CUDA di Google Colab

1. Di Mac, pull kode terbaru lalu jalankan `python3 scripts/cuda_reference.py --export`.
   Simpan lokasi ZIP yang dicetak; tidak ada inferensi baru di Mac.
2. Buka [Google Colab](https://colab.research.google.com/) → File → Upload notebook,
   pilih **`notebooks/09_colab_reference.ipynb`** dari checkout Mac.
3. Runtime → Change runtime type → GPU (misalnya T4 jika tersedia), lalu Save.
4. Run all. Sel pertama memeriksa GPU dan meminta **satu ZIP** hasil ekspor Mac.
   Upload ZIP tersebut, bukan file audio/ZIP feedback yang lama.
5. Notebook menyiapkan runtime/bobot dan menjalankan satu B: Base, seed 0, CFG 7,
   50 langkah, 30 detik. Hasil `cuda_result.zip` diunduh pada sel terakhir.

Tidak memerlukan server pribadi, terminal Colab, token GitHub, atau mount Google
Drive. Snapshot source dan input B ada dalam ZIP; bobot model tidak ikut.
Pemasangan baru mengunduh aset sekitar 13.86 GiB plus runtime. Diperlukan ruang
sekitar 45–50 GiB. FFmpeg dipasang di VM Colab jika belum tersedia.

GPU Colab mengikuti ketersediaan akun. Sesi Colab bersifat sementara: unduh paket
hasil sebelum sesi berakhir. Menjalankan ulang paket yang sama pada VM yang sama
memakai folder/runtime/cache yang sudah ada, dan hasil inferensi tetap ke folder
baru. Tidak menyewa atau membeli GPU otomatis.

CUDA dapat memakai dtype/RNG berbeda dari MPS meski seed sama. Report mencatat
perangkat dan dtype aktual; satu hasil belum membuktikan bug MPS. Untuk hanya
memeriksa paket/plan, set `DRY_RUN = True` pada sel pertama.


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, json, shutil, stat, subprocess, uuid, zipfile
DRY_RUN = False
BUNDLE_ZIP = None  # Biarkan None di Colab: tombol upload akan muncul.
if not DRY_RUN:
    from google.colab import files
    assert shutil.which("nvidia-smi"), "Pilih Runtime → Change runtime type → GPU, lalu jalankan ulang."
if BUNDLE_ZIP is None:
    from google.colab import files
    uploaded = files.upload()
    assert len(uploaded) == 1, "Upload tepat satu cuda_reference_*.zip hasil ekspor Mac"
    BUNDLE_ZIP = Path(next(iter(uploaded))).resolve()
BUNDLE_ZIP = Path(BUNDLE_ZIP).expanduser().resolve()
assert BUNDLE_ZIP.suffix.lower() == ".zip", "Pilih ZIP hasil ekspor Mac"
with zipfile.ZipFile(BUNDLE_ZIP) as archive:
    entries = archive.infolist()
    names = [entry.filename for entry in entries]
    assert len(names) == len(set(names)), "Paket memiliki nama file ganda"
    assert sum(entry.file_size for entry in entries) <= 256*1024**2, "Paket terlalu besar; gunakan exporter tanpa bobot"
    assert "bundle.json" in names, "Gunakan cuda_reference_*.zip hasil exporter, bukan feedback.zip"
    bundle_bytes = archive.read("bundle.json")
    manifest = json.loads(bundle_bytes)
    assert manifest.get("schema_version") == 1, "Paket CUDA tidak didukung"
    expected = {record["path"] for record in manifest["files"]} | {"bundle.json"}
    assert set(names) == expected, "Isi ZIP tidak cocok dengan manifest paket CUDA"
    destination_key = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()[:16]
    COLAB_BUNDLE = (Path.cwd() / ("songmap_cuda_" + destination_key)).resolve()
    data_to_write = {}
    for entry in entries:
        relative = PurePosixPath(entry.filename)
        assert not relative.is_absolute() and ".." not in relative.parts and "\\" not in entry.filename, "Path ZIP tidak valid"
        assert not stat.S_ISLNK(entry.external_attr >> 16), "Paket tidak boleh memuat symlink"
        target = (COLAB_BUNDLE / entry.filename).resolve()
        assert target.is_relative_to(COLAB_BUNDLE), "Path paket keluar dari folder tujuan"
    for record in manifest["files"]:
        data = archive.read(record["path"])
        assert hashlib.sha256(data).hexdigest() == record["sha256"], f"Paket rusak: {record['path']}"
        data_to_write[record["path"]] = data
    data_to_write["bundle.json"] = bundle_bytes
    for name, data in data_to_write.items():
        target = COLAB_BUNDLE / name
        if target.exists():
            assert target.is_file() and hashlib.sha256(target.read_bytes()).digest() == hashlib.sha256(data).digest(), f"Snapshot lokal berubah: {name}"
    COLAB_BUNDLE.mkdir(parents=True, exist_ok=True)
    for name, data in data_to_write.items():
        target = COLAB_BUNDLE / name
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(data)
print("Paket diverifikasi:", COLAB_BUNDLE)
print("Commit snapshot:", manifest["bridge_commit"])
print("Input B SHA256:", manifest["source_sha256"])


In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, sys, uuid
from IPython.display import Audio, display
ROOT = COLAB_BUNDLE / "source"
BUNDLE = ROOT.parent
assert (ROOT / "scripts/cuda_reference.py").is_file(), "Buka notebook dari source/notebooks pada paket ekspor"
manifest = json.loads((BUNDLE / "bundle.json").read_text())
assert manifest["schema_version"] == 1
for record in manifest["files"]:
    path = (BUNDLE / record["path"]).resolve()
    assert path.is_relative_to(BUNDLE.resolve()) and path.is_file(), "Path paket tidak valid"
    if "notebook_source_sha256" in record:
        book = json.loads(path.read_text())
        cells = [{"cell_type": c["cell_type"], "source": ("".join(c["source"]) if isinstance(c["source"], list) else c["source"]).replace("DRY_RUN = True", "DRY_RUN = False")} for c in book["cells"]]
        actual = hashlib.sha256(json.dumps(cells, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
        expected = record["notebook_source_sha256"]
    else:
        actual, expected = hashlib.sha256(path.read_bytes()).hexdigest(), record["sha256"]
    assert actual == expected, f"File paket berubah: {record['path']}"
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "tools/stem_eval"))
from cuda_reference import check_cuda, run_reference
from runtime_environment import runtime_environment
from setup_runtime import runtime_python, UPSTREAM
from stream_process import run_logged
from system_tools import require_audio_tools
if not DRY_RUN and any(shutil.which(name) is None for name in ("ffmpeg", "ffprobe")):
    print("Memasang ffmpeg pada VM Colab…")
    run_logged(["apt-get", "update"], check=True)
    run_logged(["apt-get", "install", "-y", "ffmpeg"], check=True)
require_audio_tools()
if not DRY_RUN:
    assert sys.platform.startswith("linux"), "Notebook referensi ini untuk server Linux CUDA"
    assert shutil.which("nvidia-smi"), "GPU NVIDIA tidak terlihat. Pilih server/kernel GPU sebelum instalasi."
    print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], text=True))
missing = sum(item["size"] for item in json.loads((ROOT / "configs/assets-base.json").read_text())["files"] if not (UPSTREAM / item["destination"]).is_file())
required = 2*missing + 5*1024**3 + (8*1024**3 if not runtime_python().is_file() else 0)
print("Commit snapshot:", manifest["bridge_commit"])
print("Input B SHA256:", manifest["source_sha256"])
print("Ruang kosong GiB:", round(shutil.disk_usage(ROOT).free/1024**3, 1), "| Estimasi ruang baru GiB:", round(required/1024**3, 1))
if not DRY_RUN:
    assert shutil.disk_usage(ROOT).free >= required, "Ruang disk belum cukup untuk runtime/cache dan bobot baru"
REFERENCE = BUNDLE / "reference"
WORK = ROOT / "runs" / ("cuda_reference_" + uuid.uuid4().hex[:12])
WORK.parent.mkdir(exist_ok=True)


## 1. Runtime pinned, terpisah dari kernel Jupyter

Jika runtime model sudah ada, gunakan dan verifikasi CUDA. Jika belum, setup
meng-clone upstream dan memasang dependency dari lockfile. PyTorch di kernel
notebook tidak dipakai untuk generasi. Tidak beralih diam-diam ke CPU.


In [ ]:
if DRY_RUN:
    print("DRY RUN: instalasi dan CUDA runtime check dilewati")
else:
    if not runtime_python().is_file():
        run_logged([sys.executable, str(ROOT / "scripts/setup_runtime.py"), "--install"], env=runtime_environment(), check=True)
    print(json.dumps(check_cuda(runtime_python()), indent=2))


## 2. Bobot pretrained yang dipin

Paket transfer tidak memuat bobot. Pada server baru, aset harus diunduh atau
sudah disalin ke `source/vendor/ACE-Step-1.5/checkpoints/`. Unduhan dan verifikasi
menampilkan progress; file lengkap yang valid tidak diunduh ulang.


In [ ]:
if DRY_RUN:
    print("DRY RUN: tidak ada bobot diunduh atau model dimuat")
else:
    run_logged([str(runtime_python()), str(ROOT / "scripts/model_assets.py"),
                "--manifest", str(ROOT / "configs/assets-base.json"),
                "--upstream", str(UPSTREAM), "--download"], env=runtime_environment(), check=True)


## 3. Satu generasi CUDA dengan B dan konfigurasi baseline

Input tetap byte-identik, 32 kHz PCM16 yang sama dengan uji MPS. Ada resampling
internal upstream; native 48 kHz baru diuji setelah baseline ini, agar format
input tidak ikut menjadi variabel. BPM/key/time signature/chord/beat belum
menjadi input model pada pembanding ini. Report menyimpan perangkat, versi,
dtype aktual dan parameter yang benar-benar dipakai. Error/artefak bukan bukti
model berhasil hanya karena proses selesai.


In [ ]:
report = run_reference(REFERENCE, WORK, execute=not DRY_RUN)
print(json.dumps(report, indent=2))
print("Folder hasil:", WORK)


## 4. Bandingkan, lalu ekspor hasil

Dengarkan `mps_baseline_listen.wav` dan `cuda_listen.wav` pada perangkat dan
volume yang sama. Keduanya memakai gain 0.7. Jika CUDA bersih, perbedaan runtime/
dtype/RNG perlu ditelusuri; jika CUDA juga rusak, fokus pada model/input/
conditioning dan evaluasi kualitas. Sweep seed/CFG, metadata SongMap dan durasi
60–180 detik menunggu hasil satu pembanding ini.


In [ ]:
if report.get("status") == "generated":
    for name in ("mps_baseline_listen.wav", "cuda_listen.wav"):
        print(name)
        display(Audio(filename=str(WORK / name)))
    print("Decoder CUDA aktual:", json.dumps(report.get("decoder_capture", {}), indent=2))
else:
    print("Belum ada hasil inferensi CUDA nyata; status:", report.get("status"))


In [ ]:
import zipfile
result_zip = WORK / "cuda_result.zip"
paths = [WORK / name for name in ("cuda_reference.json", "session.json", "mps_baseline_listen.wav", "cuda_listen.wav")]
paths += list((WORK / "capture").glob("*.json")) + list((WORK / "capture").glob("*.npy"))
paths += list((WORK / "generation").glob("*/result.json"))
with zipfile.ZipFile(result_zip, "x", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in paths:
        if path.is_file():
            archive.write(path, arcname=str(path.relative_to(WORK)))
print("Paket hasil tanpa bobot model:", result_zip)


## Unduh hasil sebelum sesi berakhir

ZIP hasil hanya berisi audio/report/latent kecil, tanpa bobot model.


In [ ]:
if DRY_RUN:
    print("DRY RUN: tidak ada unduhan browser atau hasil CUDA nyata")
else:
    from google.colab import files
    files.download(str(result_zip))
